# 14 层次聚类 Hierarchical Clustering

依赖安装说明：`pip install numpy matplotlib scikit-learn`

层次聚类会构造样本之间的层级关系。凝聚式层次聚类从每个点都是一个簇开始，不断合并最近的簇。


## 1. 数学逻辑

凝聚式聚类流程：

1. 初始时每个样本都是一个簇。
2. 计算簇之间距离。
3. 合并距离最近的两个簇。
4. 重复直到达到目标簇数。

常见 linkage：

- single：两个簇中最近点的距离。
- complete：两个簇中最远点的距离。
- average：所有点对距离平均。
- ward：合并后簇内方差增加最小。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering

np.random.seed(42)
X, _ = make_blobs(n_samples=90, centers=3, cluster_std=0.75, random_state=42)


In [ ]:
# 从零实现：非常小规模的 average linkage 合并过程

def cluster_distance(X, c1, c2):
    distances = []
    for i in c1:
        for j in c2:
            distances.append(np.sqrt(np.sum((X[i] - X[j]) ** 2)))
    return np.mean(distances)

clusters = [[i] for i in range(12)]  # 只用前 12 个点演示，避免输出太长
X_small = X[:12]
while len(clusters) > 3:
    best = None
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            dist = cluster_distance(X_small, clusters[i], clusters[j])
            if best is None or dist < best[0]:
                best = (dist, i, j)
    dist, i, j = best
    clusters[i] = clusters[i] + clusters[j]
    clusters.pop(j)
    print(f'合并后簇数量={len(clusters):2d} | 本次距离={dist:.3f}')

print('最终小样本簇:', clusters)


In [ ]:
for linkage in ['ward', 'complete', 'average', 'single']:
    model = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    labels = model.fit_predict(X)
    plt.figure()
    plt.scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=28)
    plt.title(f'AgglomerativeClustering linkage={linkage}')
    plt.show()


## 2. 常见误区

- 层次聚类计算成本较高，大数据集上不一定合适。
- linkage 选择会显著影响结果。
- 它能给出层级结构，但最终切成几类仍然需要决策。

## 3. 小实验

- 改 `n_clusters`。
- 对比不同 linkage。
- 增大样本数，感受运行时间变化。
